# Sentiment Analysis using NLP & ML Models

## Pipeline
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison

## 1. Data Understanding

We load the dataset and analyze:
- Number of samples
- Class distribution
- Sample text entries

# Import & Load Data

In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('Twitter_Data.csv')

# Rename columns for convenience
df = df.rename(columns={
    'clean_text': 'text',
    'category': 'label'
})

print(df.head())
print("\nShape:", df.shape)

                                                text  label
0  when modi promised “minimum government maximum...   -1.0
1  talk all the nonsense and continue all the dra...    0.0
2  what did just say vote for modi  welcome bjp t...    1.0
3  asking his supporters prefix chowkidar their n...    1.0
4  answer who among these the most powerful world...    1.0

Shape: (162980, 2)


#  Data Understanding

In [3]:
# Class distribution
print(df['label'].value_counts())

# Check null values
print(df.isnull().sum())

label
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64
text     4
label    7
dtype: int64


# Handle Missing Values

In [4]:
df.dropna(inplace=True)

# NLP Preprocessing
## Preprocessing
We clean text using:
- Lowercasing
- Removing URLs
- Removing punctuation
- Removing stopwords
- Stemming

In [5]:
import sys
!{sys.executable} -m pip install nltk

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    words = [stemmer.stem(w) for w in words if w not in stop_words]

    return " ".join(words)

# Apply preprocessing
df['cleaned'] = df['text'].apply(preprocess)

print(df[['text', 'cleaned']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                                text  \
0  when modi promised “minimum government maximum...   
1  talk all the nonsense and continue all the dra...   
2  what did just say vote for modi  welcome bjp t...   
3  asking his supporters prefix chowkidar their n...   
4  answer who among these the most powerful world...   

                                             cleaned  
0  modi promis minimum govern maximum govern expe...  
1               talk nonsens continu drama vote modi  
2  say vote modi welcom bjp told rahul main campa...  
3  ask support prefix chowkidar name modi great s...  
4  answer among power world leader today trump pu...  


# Feature Engineering

In [7]:
import sys
!{sys.executable} -m pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['cleaned'])
y = df['label']

# Train-Test Split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Building

## Logistic Regression

In [10]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

## Naive Bayes

In [11]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

## Decision Tree

In [12]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## Evaluation

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)


Logistic Regression
Accuracy: 0.8434681229674174
Precision: 0.8443328689282199
Recall: 0.8434681229674174
F1 Score: 0.8421680546522017

Naive Bayes
Accuracy: 0.6869055654414923
Precision: 0.723766533031474
Recall: 0.6869055654414923
F1 Score: 0.6719438249177822

Decision Tree
Accuracy: 0.7695894949990796
Precision: 0.7682345434742358
Recall: 0.7695894949990796
F1 Score: 0.7687812784894158


## Comparison & Insights

- Logistic Regression performed best due to handling sparse TF-IDF features efficiently.
- Naive Bayes performed well and is fast but assumes feature independence.
- Decision Tree performed lower due to overfitting.

### Best Preprocessing:
- Stopword removal + stemming improved performance

### Best Feature Engineering:
- TF-IDF performed better than Bag of Words

### Final Conclusion:
TF-IDF + Logistic Regression is the best combination for this dataset.

In [14]:
def predict_sentiment(text):
    cleaned = preprocess(text)
    vector = tfidf.transform([cleaned])
    return lr.predict(vector)[0]

print(predict_sentiment("I love this product"))

1.0
